In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv("../data/fraudTrain_prepared.csv")
print(df.shape)
print(df.columns.tolist())

(1296675, 25)
['trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud', 'hour_of_day', 'day_of_week', 'month']


In [3]:
features = ['amt', 'category', 'gender', 'city_pop', 'hour_of_day', 'day_of_week', 'month', 'lat', 'long', 'merch_lat', 'merch_long']

X= df[features].copy()
y = df['is_fraud'].copy()

# One hot encoding for categorical features
cat_cols = ['category' ,'gender', 'day_of_week', 'month']
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print(f"Final Features: {X.columns.tolist()}")

Final Features: ['amt', 'city_pop', 'hour_of_day', 'lat', 'long', 'merch_lat', 'merch_long', 'category_food_dining', 'category_gas_transport', 'category_grocery_net', 'category_grocery_pos', 'category_health_fitness', 'category_home', 'category_kids_pets', 'category_misc_net', 'category_misc_pos', 'category_personal_care', 'category_shopping_net', 'category_shopping_pos', 'category_travel', 'gender_M', 'day_of_week_1', 'day_of_week_2', 'day_of_week_3', 'day_of_week_4', 'day_of_week_5', 'day_of_week_6', 'month_2', 'month_3', 'month_4', 'month_5', 'month_6', 'month_7', 'month_8', 'month_9', 'month_10', 'month_11', 'month_12']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Fraud rate - Train: {y_train.mean():.4%} | Test: {y_test.mean():.4%}")

Train: (1037340, 38) | Test: (259335, 38)
Fraud rate - Train: 0.5789% | Test: 0.5788%


# Why SMOTE?

* Handles severe class imbalance (0.58% fraud)
* Generates synthetic fraud samples
* Applied only on train set to prevent data leakage

In [5]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42, sampling_strategy='auto')

X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print(f"After SMOTE:")
print(f"Original train shape : {X_train.shape}")
print(f"Resampled train shape: {X_train_smote.shape}")
print(f"New fraud rate       : {y_train_smote.mean():.4%}")

After SMOTE:
Original train shape : (1037340, 38)
Resampled train shape: (2062670, 38)
New fraud rate       : 50.0000%


---

# Baseline models

**Why Logistic Regression ?**

* Simple, Fast, Interpretable baseline and good for understanding linear relationships

**Why Decision Tree**

* Captures no-linear patterns

**Why Random Forest**

* Handles imbalance well and reduces overfitting

**Why XGBoost**

* Strong baseline, State-of-the-art for fraud detection. Excellent with imbalance via scale_pos_weight

In [9]:
models = {}

# Logistic Regression
models['Logistic Regression'] = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# Decision Tree
models['Decision Tree'] = DecisionTreeClassifier(
    max_depth = 8,
    class_weight='balanced',
    random_state = 42
)

# Random Forest
models['Random Forest'] = RandomForestClassifier(
    n_estimators = 150,
    max_depth=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# XGBoost
models['XGBoost'] = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate= 0.1,
    scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1]),
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False
)

**Training and Model Comparision**

In [10]:
# Training and evaluation
results = []

for name, model in models.items():
    print(f"Training {name} :")
    model.fit(X_train_smote, y_train_smote)

    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:,1] if hasattr(model, "predict_proba") else None

    report = classification_report(y_test, pred, output_dict=True)

    results.append({
        'Model' : name,
        'Precision' : report['1']['precision'],
        'Recall' : report['1']['recall'],
        'F1-Score' : report['1']['f1-score'],
        'PR-AUC' : average_precision_score(y_test, proba) if proba is not None else None,
        'ROC-AUC' : roc_auc_score(y_test, proba) if proba is not None else None
    })

# Comparision 

comparision_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("BASELINE MODELS COMPARISON")
print("="*70) 
display(comparision_df.round(4).sort_values(by='F1-Score', ascending = False))

Training Logistic Regression :
Training Decision Tree :
Training Random Forest :
Training XGBoost :



,Model,Precision,Recall,F1-Score,PR-AUC,ROC-AUC
2,Random Forest,0.3022,0.8321,0.4434,0.7351,0.9867
1,Decision Tree,0.2286,0.9207,0.3662,0.7232,0.9836
3,XGBoost,0.1156,0.9793,0.2068,0.8486,0.9958
0,Logistic Regression,0.0855,0.6689,0.1517,0.1612,0.8798


**Model-by-Model Analysis**<br>
Random Forest 🌲

Highest F1-score (0.4434) → best balance between precision and recall.

Precision = 0.3022 → fewer false alarms compared to other models.

Recall = 0.8321 → still detects most fraud.

📌 Conclusion: Best overall practical model among the four.

Decision Tree 🌳

Recall = 0.9207 → detects most fraud cases.

Precision lower than Random Forest.

F1-score moderate.

📌 Conclusion: Good if detecting fraud is the top priority.

XGBoost ⚡

Recall = 0.9793 (very high) → catches almost all fraud.

Precision extremely low (0.1156) → many legitimate transactions flagged as fraud.

PR-AUC highest (0.8486) which is good for imbalanced datasets.

📌 Conclusion: Very sensitive model but causes too many false positives.

Logistic Regression 📉

Lowest scores in almost all metrics.

PR-AUC especially poor.

📌 Conclusion: Not suitable for this dataset.

3. Why ROC-AUC is Misleading Here

All models show very high ROC-AUC (>0.98) except Logistic Regression.

But in fraud detection, ROC-AUC can be misleading because:

Dataset is highly imbalanced

ROC considers true negatives heavily

Fraud detection cares more about precision and recall

That is why PR-AUC is more important, and here XGBoost and Random Forest perform best.

4. Practical Model Choice (Fraud Detection)

Depends on business goal:

If minimizing missed fraud is critical

Choose XGBoost

Recall = 0.9793

If balance between false alarms and fraud detection is needed

Choose Random Forest ✅

Best F1-score

If recall is important but want simpler model

Choose Decision Tree

5. Final Ranking (Practical Use)

1️⃣ Random Forest – Best balance (Recommended)
2️⃣ XGBoost – Best detection but too many false alarms
3️⃣ Decision Tree – Good recall but weaker precision
4️⃣ Logistic Regression – Poor performance

✅ Conclusion:
After applying SMOTE, Random Forest is the best performing model overall, while XGBoost achieves the highest fraud detection rate but with many false positives.